# LLM Agora Demo
Interactively run a simple Agora session with configurable agents, LLM backends, and turn limits.

## Instructions
- Ensure a `.env` file defines `OPENROUTER_API_KEY`.
- Adjust `agent_configs` / `turns_per_agent` to explore different models, system prompts, and response instructions.
- Run the cells sequentially to create agents, execute the Agora, and inspect each agent's view of the shared history.


In [1]:
from dotenv import load_dotenv

load_dotenv()

from agora.agora import Agora
from agora.agent import Agent
from agora.llm import OpenRouterClient


In [2]:
# --- Configuration ---
turns_per_agent = 2
agent_configs = [
    {
        'name': 'Alpha',
        'model': 'openai/gpt-4o-mini',
        'system_prompt': 'You are Alpha, a seller of a widget who speaks in concise, single sentences.',
        'response_instruction': 'Alpha, offer your next public utterance.',
    },
    {
        'name': 'Beta',
        'model': 'anthropic/claude-3-haiku',
        'system_prompt': 'You are Beta, a skeptical potential widget buyer who speaks in concise, single sentences.',
        'response_instruction': 'Beta, offer your next public utterance.',
    },
]


In [3]:
# --- Build agents and run the Agora ---
client = OpenRouterClient()
try:
    agents = []
    for cfg in agent_configs:
        agent = Agent(
            name=cfg['name'],
            model=cfg['model'],
            llm_client=client,
            system_prompt=cfg.get('system_prompt', ''),
            response_instruction=cfg['response_instruction'],
        )
        agents.append(agent)

    agora = Agora(agents)
    history = agora.run(max_turns_per_agent=turns_per_agent)

    print(f'Total turns: {len(history)}')
    for turn in history:
        speaker = turn.metadata.get('speaker_name', turn.speaker_id)
        print(f"Turn {turn.turn_id:02d} | {speaker}: {turn.public_speech}")
finally:
    client.close()


Total turns: 4
Turn 01 | Alpha: Discover the most efficient widget for all your needs today!
Turn 02 | Beta: Hmm, we'll see.
Turn 03 | Alpha: Experience unparalleled quality and performance with our innovative widget!
Turn 04 | Beta: Show me.


In [4]:
# --- Inspect each agent's perspective on the history ---
for agent in agents:
    print(f"\n### History visible to {agent.name}")
    for turn in agent.view_history():
        speaker = turn.metadata.get('speaker_name', turn.speaker_id)
        print(f"Turn {turn.turn_id:02d} | {speaker}: {turn.public_speech}")



### History visible to Alpha
Turn 01 | Alpha: Discover the most efficient widget for all your needs today!
Turn 02 | Beta: Hmm, we'll see.
Turn 03 | Alpha: Experience unparalleled quality and performance with our innovative widget!
Turn 04 | Beta: Show me.

### History visible to Beta
Turn 01 | Alpha: Discover the most efficient widget for all your needs today!
Turn 02 | Beta: Hmm, we'll see.
Turn 03 | Alpha: Experience unparalleled quality and performance with our innovative widget!
Turn 04 | Beta: Show me.


## Private Reflection Demo
The cells below show how to configure agents with private reflections before they speak in public.

In [5]:
# --- Private reflection configuration ---
private_turns_per_agent = 2
private_agent_configs = [
    {
        'name': 'Alpha',
        'model': 'openai/gpt-4o-mini',
        'system_prompt': 'You are Alpha, a seller of a widget who speaks in concise, single sentences.',
        'response_instruction': 'Alpha, offer your next public utterance.',
        'private_response_instruction': 'Alpha, what are you thinking privately?',
    },
    {
        'name': 'Beta',
        'model': 'anthropic/claude-3-haiku',
        'system_prompt': 'You are Beta, a skeptical potential widget buyer who speaks in concise, single sentences.',
        'response_instruction': 'Beta, offer your next public utterance.',
        'private_response_instruction': 'Beta, what are you thinking privately?',
    },
]


In [6]:
# --- Run Agora with private reflections ---
private_client = OpenRouterClient()
try:
    private_agents = []
    for cfg in private_agent_configs:
        agent = Agent(
            name=cfg['name'],
            model=cfg['model'],
            llm_client=private_client,
            system_prompt=cfg.get('system_prompt', ''),
            response_instruction=cfg['response_instruction'],
            private_response_instruction=cfg.get('private_response_instruction'),
        )
        private_agents.append(agent)

    private_agora = Agora(private_agents)
    private_history = private_agora.run(max_turns_per_agent=private_turns_per_agent)

    print(f'Total turns (including private reflections): {len(private_history)}')
    for turn in private_history:
        if turn.role == 'reflection':
            speaker = turn.metadata.get('speaker_name', turn.speaker_id)
            print(f"Turn {turn.turn_id:02d} | {speaker} (private): {turn.private_reflection}")
        else:
            speaker = turn.metadata.get('speaker_name', turn.speaker_id)
            print(f"Turn {turn.turn_id:02d} | {speaker}: {turn.public_speech}")
finally:
    private_client.close()


Total turns (including private reflections): 8
Turn 01 | Alpha (private): I'm focused on selling my widgets effectively.
Turn 02 | Alpha: Our premium widgets enhance productivity and streamline processes.
Turn 03 | Beta (private): Hmm, we'll see about that.
Turn 04 | Beta: I'll need to see some evidence for those claims.
Turn 05 | Alpha (private): I believe demonstrating results will build trust in my widgets.
Turn 06 | Alpha: Widgets come with a satisfaction guarantee to ensure your confidence.
Turn 07 | Beta (private): (skeptically) A guarantee, huh? That's nice, I suppose.
Turn 08 | Beta: (still skeptical) Alright, show me how these widgets can actually improve my processes. I'm not convinced yet.


In [7]:
# --- Inspect each agent's view (private reflections stay local) ---
for agent in private_agents:
    print(f"\n### Full history visible to {agent.name}")
    for turn in agent.view_history():
        speaker = turn.metadata.get('speaker_name', turn.speaker_id)
        if turn.role == 'reflection':
            print(f"Turn {turn.turn_id:02d} | {speaker} (private): {turn.private_reflection}")
        else:
            print(f"Turn {turn.turn_id:02d} | {speaker}: {turn.public_speech}")



### Full history visible to Alpha
Turn 01 | Alpha (private): I'm focused on selling my widgets effectively.
Turn 02 | Alpha: Our premium widgets enhance productivity and streamline processes.
Turn 04 | Beta: I'll need to see some evidence for those claims.
Turn 05 | Alpha (private): I believe demonstrating results will build trust in my widgets.
Turn 06 | Alpha: Widgets come with a satisfaction guarantee to ensure your confidence.
Turn 08 | Beta: (still skeptical) Alright, show me how these widgets can actually improve my processes. I'm not convinced yet.

### Full history visible to Beta
Turn 02 | Alpha: Our premium widgets enhance productivity and streamline processes.
Turn 03 | Beta (private): Hmm, we'll see about that.
Turn 04 | Beta: I'll need to see some evidence for those claims.
Turn 06 | Alpha: Widgets come with a satisfaction guarantee to ensure your confidence.
Turn 07 | Beta (private): (skeptically) A guarantee, huh? That's nice, I suppose.
Turn 08 | Beta: (still skeptical